# 异常

## RejectedOrderError
当订单因各种原因被投资组合模拟系统拒绝时抛出的异常。
```python
class RejectedOrderError(Exception):
    pass
```

# 核心配置

## InitCashMode
初始资金模式
- Auto (0): 自动模式
  - 模拟过程中资金视为无限
  - 模拟结束后设置为实际花费的总金额
- AutoAlign (1): 自动对齐模式  
  - 设置为所有列（资产）总花费金额的统一值

```python
class InitCashModeT(tp.NamedTuple):
    Auto: int = 0
    AutoAlign: int = 1   # 自动对齐模式：设置为所有列总花费金额的对齐值

InitCashMode = InitCashModeT()
```


## CallSeqType
调用序列类型定义，控制投资组合模拟中多个资产（列）的处理顺序。
- Default (0): 默认顺序
  - 按列的自然顺序从左到右处理
- Reversed (1): 反向顺序
  - 按列的相反顺序从右到左处理  
- Random (2): 随机顺序
  - 每个时间步随机打乱处理顺序
  - 消除顺序偏差，提供更公平的资金分配
- Auto (3): 自动顺序
  - 根据订单的价值动态排序处理
  - 卖单优先执行以释放资金供买单使用
  - 实现更智能的资金利用，但可能引入前瞻偏差

```python
class CallSeqTypeT(tp.NamedTuple):
    Default: int = 0
    Reversed: int = 1
    Random: int = 2
    Auto: int = 3

CallSeqType = CallSeqTypeT()
```

## AccumulationMode
仓位累积模式类型定义，控制投资组合中仓位的累积行为。
- Disabled (0): 禁用累积
    - 不允许任何形式的仓位累积
    - 每次信号都会完整执行，不考虑现有仓位
    - 适用于简单的买入-持有-卖出策略
- Both (1): 双向累积
    - 允许在现有仓位基础上继续增加或减少
    - 支持分批建仓和分批减仓
- AddOnly (2): 仅增加模式
    - 只允许在现有仓位基础上增加
    - 不允许减少现有仓位
    - 适用于趋势跟踪和动量策略
- RemoveOnly (3): 仅减少模式
    - 只允许减少现有仓位
    - 不允许增加现有仓位
    - 适用于止盈和风险控制策略

```python
class AccumulationModeT(tp.NamedTuple):
    Disabled: int = 0
    Both: int = 1
    AddOnly: int = 2
    RemoveOnly: int = 3

AccumulationMode = AccumulationModeT()
```

## ConflictMode 和 DirectionConflictMode
| 特征 | `ConflictModeT` | `DirectionConflictModeT` |
| :--- | :--- | :--- |
| **处理的冲突类型** | **动作冲突**：<br>`入场(Entry)` vs `出场(Exit)` 信号冲突。 | **方向冲突**：<br>`多头入场(Long Entry)` vs `空头入场(Short Entry)` 信号冲突。 |
| **解决的问题** | “在当前时间点，我应该**开/加仓**还是**平/减仓**？” | “在当前时间点，我应该**开多仓**还是**开空仓**？” |
| **Ignore (忽略)** | 忽略冲突的入场和出场信号，维持仓位不变。 | 忽略冲突的多头和空头入场信号，维持仓位不变。 |
| **Entry / Long (优先)** | **(Entry)**<br>优先执行**入场**信号，忽略出场信号。<br>**效果**: 建立或增加当前方向的仓位。 | **(Long)**<br>优先执行**多头入场**信号，忽略空头入场信号。<br>**效果**: 建立或增加**多头**仓位。 |
| **Exit / Short (优先)** | **(Exit)**<br>优先执行**出场**信号，忽略入场信号。<br>**效果**: 平仓或减少当前方向的仓位。 | **(Short)**<br>优先执行**空头入场**信号，忽略多头入场信号。<br>**效果**: 建立或增加**空头**仓位。 |
| **Adjacent (相邻优先)** | **执行与当前状态“相邻”的操作。**<br>该模式下会同时保留入场和出场信号，最终行为取决于 `accumulate` 参数。 (详见下表) | **执行与当前持仓方向相同的入场信号。**<br>**持多仓时**: 执行多头入场(顺势加仓)。<br>**持空仓时**: 执行空头入场(顺势加仓)。<br>**无仓位时**: 忽略所有信号。 |
| **Opposite (相反优先)** | **执行与当前状态“相反”的操作。**<br>例如，持多仓时优先执行平仓信号，为反转做准备。 | **执行与当前持仓方向相反的入场信号。**<br>**持多仓时**: 执行空头入场(逆势反转)。<br>**持空仓时**: 执行多头入场(逆势反转)。<br>**无仓位时**: 忽略所有信号。 |

| `ConflictMode.Adjacent` 与... | 信号处理 | 最终动作 | 仓位变化 | 核心逻辑 |
| :--- | :--- | :--- | :--- | :--- |
| **`accumulate=False`** <br> (默认) | 冲突发生时，`is_entry` 和 `is_exit` 都被保留为 `True`，但累积模式不允许信号叠加。 | **执行 `Exit` (出场)**，因为在非累积模式下，出场指令优先于入场指令。 | **减少** | **风控优先**: 在考虑增仓前，先执行减仓指令。 |
| **`accumulate=True`** <br> (开启累积) | 冲突发生时，`is_entry` 和 `is_exit` 都被保留为 `True`，且累积模式允许信号叠加。 | **同时执行 `Entry` 和 `Exit`**，一个增仓信号和一个减仓信号的效果相互抵消。 | **不变** | **信号叠加**: 允许正负信号同时作用，最终效果为净变化。 |


```python
class ConflictModeT(tp.NamedTuple):
    Ignore: int = 0
    Entry: int = 1
    Exit: int = 2
    Adjacent: int = 3
    Opposite: int = 4

ConflictMode = ConflictModeT()

class DirectionConflictModeT(tp.NamedTuple):
    Ignore: int = 0
    Long: int = 1
    Short: int = 2
    Adjacent: int = 3
    Opposite: int = 4

DirectionConflictMode = DirectionConflictModeT()
```

## OppositeEntryMode
| 模式 | 触发条件 | 信号处理 | 累积模式影响 | 最终效果 | 适用场景 |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Ignore** <br> (忽略) | 持有多头时收到空头入场信号<br>持有空头时收到多头入场信号 | **完全忽略**反向入场信号<br>保持原有信号不变 | 不影响累积模式 | **保持当前仓位**<br>等待正常退出信号 | **坚持方向策略**：<br>- 趋势跟踪策略<br>- 避免频繁换向<br>- 长期持仓策略 |
| **Close** <br> (直接平仓) | 持有多头时收到空头入场信号<br>持有空头时收到多头入场信号 | **取消**反向入场信号<br>**生成**对应的平仓信号 | **强制禁用**累积模式<br>`accumulate = Disabled` | **立即平仓**<br>回到空仓状态<br>不建立反向仓位 | **保守策略**：<br>- 风险规避<br>- 不确定时先平仓<br>- 防止过度交易 |
| **CloseReduce** <br> (平仓减少) | 持有多头时收到空头入场信号<br>持有空头时收到多头入场信号 | **取消**反向入场信号<br>**生成**对应的平仓信号 | **保持**原累积模式<br>不修改 `accumulate` | **平仓或减仓**<br>- `accumulate=False`: 完全平仓<br>- `accumulate=True`: 按信号大小减仓 | **灵活调仓策略**：<br>- 分批平仓<br>- 渐进式退出<br>- 部分获利了结 |
| **Reverse** <br> (完全反转) | 持有多头时收到空头入场信号<br>持有空头时收到多头入场信号 | **保持**反向入场信号<br>不生成额外的平仓信号 | **强制禁用**累积模式<br>`accumulate = Disabled` | **直接反转**<br>从多头→空头<br>或从空头→多头 | **快速反转策略**：<br>- 趋势反转交易<br>- 快速响应市场<br>- 动量策略 |
| **ReverseReduce** <br> (反转减少) | 持有多头时收到空头入场信号<br>持有空头时收到多头入场信号 | **保持**反向入场信号<br>不生成额外的平仓信号 | **保持**原累积模式<br>不修改 `accumulate` | **反转或渐进转换**<br>- `accumulate=False`: 直接反转<br>- `accumulate=True`: 先减仓再反转 | **渐进反转策略**：<br>- 分批方向转换<br>- 降低转换冲击<br>- 灵活仓位管理 |

| 反向入场模式 | `accumulate=False` | `accumulate=True` | 累积模式变化 |
| :--- | :--- | :--- | :--- |
| **Ignore** | 忽略反向信号，保持当前仓位 | 忽略反向信号，保持当前仓位 | **不变** |
| **Close** | 立即完全平仓，回到空仓 | 立即完全平仓，回到空仓 | **强制禁用** → `False` |
| **CloseReduce** | 立即完全平仓，回到空仓 | 按信号大小减少仓位，可能部分平仓 | **保持不变** |
| **Reverse** | 立即完全反转仓位方向 | 立即完全反转仓位方向 | **强制禁用** → `False` |
| **ReverseReduce** | 立即完全反转仓位方向 | 先减少当前仓位，完全平仓后建立反向仓位 | **保持不变** |


```python
class OppositeEntryModeT(tp.NamedTuple):
    Ignore: int = 0
    Close: int = 1
    CloseReduce: int = 2
    Reverse: int = 3
    ReverseReduce: int = 4
```

## StopEntryPrice
```python
class StopEntryPriceT(tp.NamedTuple):
    ValPrice: int = 0
    Price: int = 1
    FillPrice: int = 2
    Close: int = 3
```

## StopExitPrice
```python
class StopExitPriceT(tp.NamedTuple):
    StopLimit: int = 0
    StopMarket: int = 1
    Price: int = 2
    Close: int = 3
```

## StopExitMode
```python
class StopExitModeT(tp.NamedTuple):
    Close: int = 0
    CloseReduce: int = 1
    Reverse: int = 2
    ReverseReduce: int = 3
```

## StopUpdateMode
```python
class StopUpdateModeT(tp.NamedTuple):
    Keep: int = 0
    Override: int = 1
    OverrideNaN: int = 2
```

# 订单和交易

## SizeType
```python
class SizeTypeT(tp.NamedTuple):
    Amount: int = 0
    Value: int = 1
    Percent: int = 2
    TargetAmount: int = 3
    TargetValue: int = 4
    TargetPercent: int = 5
```

## Direction
```python
class DirectionT(tp.NamedTuple):
    LongOnly: int = 0
    ShortOnly: int = 1
    Both: int = 2
```

## OrderStatus
```python
class OrderStatusT(tp.NamedTuple):
    Filled: int = 0
    Ignored: int = 1
    Rejected: int = 2
```

## OrderSide
```python
class OrderSideT(tp.NamedTuple):
    Buy: int = 0
    Sell: int = 1
```

## OrderStatusInfo
```python
class OrderStatusInfoT(tp.NamedTuple):
    SizeNaN: int = 0
    PriceNaN: int = 1
    ValPriceNaN: int = 2
    ValueNaN: int = 3
    ValueZeroNeg: int = 4
    SizeZero: int = 5
    NoCashShort: int = 6
    NoCashLong: int = 7
    NoOpenPosition: int = 8
    MaxSizeExceeded: int = 9
    RandomEvent: int = 10
    CantCoverFees: int = 11
    MinSizeNotReached: int = 12
    PartialFill: int = 13
```

## TradeDirection
```python
class TradeDirectionT(tp.NamedTuple):
    Long: int = 0
    Short: int = 1
```

## TradeStatus
```python
class TradeStatusT(tp.NamedTuple):
    Open: int = 0
    Closed: int = 1
```

## TradesType
```python
class TradesTypeT(tp.NamedTuple):
    EntryTrades: int = 0
    ExitTrades: int = 1
    Positions: int = 2
```

# 状态和上下文

## ProcessOrderState
```python
class ProcessOrderState(tp.NamedTuple):
    cash: float
    position: float
    debt: float
    free_cash: float
    val_price: float
    value: float
    oidx: int
    lidx: int
```

## ExecuteOrderState
```python
class ExecuteOrderState(tp.NamedTuple):
    cash: float
    position: float
    debt: float
    free_cash: float
```

## SimulationContext
```python
class SimulationContext(tp.NamedTuple):
    target_shape: tp.Shape
    group_lens: tp.Array1d
    init_cash: tp.Array1d
    cash_sharing: bool
    call_seq: tp.Optional[tp.Array2d]
    segment_mask: tp.ArrayLike
    call_pre_segment: bool
    call_post_segment: bool
    close: tp.ArrayLike
    ffill_val_price: bool
    update_value: bool
    fill_pos_record: bool
    flex_2d: bool
    order_records: tp.RecordArray
    log_records: tp.RecordArray
    last_cash: tp.Array1d
    last_position: tp.Array1d
    last_debt: tp.Array1d
    last_free_cash: tp.Array1d
    last_val_price: tp.Array1d
    last_value: tp.Array1d
    second_last_value: tp.Array1d
    last_return: tp.Array1d
    last_oidx: tp.Array1d
    last_lidx: tp.Array1d
    last_pos_record: tp.RecordArray
```

## GroupContext
```python
class GroupContext(tp.NamedTuple):
    target_shape: tp.Shape
    group_lens: tp.Array1d
    init_cash: tp.Array1d
    cash_sharing: bool
    call_seq: tp.Optional[tp.Array2d]
    segment_mask: tp.ArrayLike
    call_pre_segment: bool
    call_post_segment: bool
    close: tp.ArrayLike
    ffill_val_price: bool
    update_value: bool
    fill_pos_record: bool
    flex_2d: bool
    order_records: tp.RecordArray
    log_records: tp.RecordArray
    last_cash: tp.Array1d
    last_position: tp.Array1d
    last_debt: tp.Array1d
    last_free_cash: tp.Array1d
    last_val_price: tp.Array1d
    last_value: tp.Array1d
    second_last_value: tp.Array1d
    last_return: tp.Array1d
    last_oidx: tp.Array1d
    last_lidx: tp.Array1d
    last_pos_record: tp.RecordArray
    group: int
    group_len: int
    from_col: int
    to_col: int
```

## RowContext
```python
class RowContext(tp.NamedTuple):
    target_shape: tp.Shape
    group_lens: tp.Array1d
    init_cash: tp.Array1d
    cash_sharing: bool
    call_seq: tp.Optional[tp.Array2d]
    segment_mask: tp.ArrayLike
    call_pre_segment: bool
    call_post_segment: bool
    close: tp.ArrayLike
    ffill_val_price: bool
    update_value: bool
    fill_pos_record: bool
    flex_2d: bool
    order_records: tp.RecordArray
    log_records: tp.RecordArray
    last_cash: tp.Array1d
    last_position: tp.Array1d
    last_debt: tp.Array1d
    last_free_cash: tp.Array1d
    last_val_price: tp.Array1d
    last_value: tp.Array1d
    second_last_value: tp.Array1d
    last_return: tp.Array1d
    last_oidx: tp.Array1d
    last_lidx: tp.Array1d
    last_pos_record: tp.RecordArray
    i: int
```

## SegmentContext
```python
class SegmentContext(tp.NamedTuple):
    target_shape: tp.Shape
    group_lens: tp.Array1d
    init_cash: tp.Array1d
    cash_sharing: bool
    call_seq: tp.Optional[tp.Array2d]
    segment_mask: tp.ArrayLike
    call_pre_segment: bool
    call_post_segment: bool
    close: tp.ArrayLike
    ffill_val_price: bool
    update_value: bool
    fill_pos_record: bool
    flex_2d: bool
    order_records: tp.RecordArray
    log_records: tp.RecordArray
    last_cash: tp.Array1d
    last_position: tp.Array1d
    last_debt: tp.Array1d
    last_free_cash: tp.Array1d
    last_val_price: tp.Array1d
    last_value: tp.Array1d
    second_last_value: tp.Array1d
    last_return: tp.Array1d
    last_oidx: tp.Array1d
    last_lidx: tp.Array1d
    last_pos_record: tp.RecordArray
    group: int
    group_len: int
    from_col: int
    to_col: int
    i: int
    call_seq_now: tp.Optional[tp.Array1d]
```

## OrderContext
```python
class OrderContext(tp.NamedTuple):
    target_shape: tp.Shape
    group_lens: tp.Array1d
    init_cash: tp.Array1d
    cash_sharing: bool
    call_seq: tp.Optional[tp.Array2d]
    segment_mask: tp.ArrayLike
    call_pre_segment: bool
    call_post_segment: bool
    close: tp.ArrayLike
    ffill_val_price: bool
    update_value: bool
    fill_pos_record: bool
    flex_2d: bool
    order_records: tp.RecordArray
    log_records: tp.RecordArray
    last_cash: tp.Array1d
    last_position: tp.Array1d
    last_debt: tp.Array1d
    last_free_cash: tp.Array1d
    last_val_price: tp.Array1d
    last_value: tp.Array1d
    second_last_value: tp.Array1d
    last_return: tp.Array1d
    last_oidx: tp.Array1d
    last_lidx: tp.Array1d
    last_pos_record: tp.RecordArray
    group: int
    group_len: int
    from_col: int
    to_col: int
    i: int
    call_seq_now: tp.Optional[tp.Array1d]
    col: int
    call_idx: int
    cash_now: float
    position_now: float
    debt_now: float
    free_cash_now: float
    val_price_now: float
    value_now: float
    return_now: float
    pos_record_now: tp.Record
```

## PostOrderContext
```python
class PostOrderContext(tp.NamedTuple):
    target_shape: tp.Shape
    group_lens: tp.Array1d
    init_cash: tp.Array1d
    cash_sharing: bool
    call_seq: tp.Optional[tp.Array2d]
    segment_mask: tp.ArrayLike
    call_pre_segment: bool
    call_post_segment: bool
    close: tp.ArrayLike
    ffill_val_price: bool
    update_value: bool
    fill_pos_record: bool
    flex_2d: bool
    order_records: tp.RecordArray
    log_records: tp.RecordArray
    last_cash: tp.Array1d
    last_position: tp.Array1d
    last_debt: tp.Array1d
    last_free_cash: tp.Array1d
    last_val_price: tp.Array1d
    last_value: tp.Array1d
    second_last_value: tp.Array1d
    last_return: tp.Array1d
    last_oidx: tp.Array1d
    last_lidx: tp.Array1d
    last_pos_record: tp.RecordArray
    group: int
    group_len: int
    from_col: int
    to_col: int
    i: int
    call_seq_now: tp.Optional[tp.Array1d]
    col: int
    call_idx: int
    cash_before: float
    position_before: float
    debt_before: float
    free_cash_before: float
    val_price_before: float
    value_before: float
    order_result: "OrderResult"
    cash_now: float
    position_now: float
    debt_now: float
    free_cash_now: float
    val_price_now: float
    value_now: float
    return_now: float
    pos_record_now: tp.Record
```

## FlexOrderContext
```python
class FlexOrderContext(tp.NamedTuple):
    target_shape: tp.Shape
    group_lens: tp.Array1d
    init_cash: tp.Array1d
    cash_sharing: bool
    call_seq: tp.Optional[tp.Array2d]
    segment_mask: tp.ArrayLike
    call_pre_segment: bool
    call_post_segment: bool
    close: tp.ArrayLike
    ffill_val_price: bool
    update_value: bool
    fill_pos_record: bool
    flex_2d: bool
    order_records: tp.RecordArray
    log_records: tp.RecordArray
    last_cash: tp.Array1d
    last_position: tp.Array1d
    last_debt: tp.Array1d
    last_free_cash: tp.Array1d
    last_val_price: tp.Array1d
    last_value: tp.Array1d
    second_last_value: tp.Array1d
    last_return: tp.Array1d
    last_oidx: tp.Array1d
    last_lidx: tp.Array1d
    last_pos_record: tp.RecordArray
    group: int
    group_len: int
    from_col: int
    to_col: int
    i: int
    call_seq_now: None
    call_idx: int
```

# 订单相关

## Order
```python
class Order(tp.NamedTuple):
    size: float = np.inf
    price: float = np.inf
    size_type: int = SizeType.Amount
    direction: int = Direction.Both
    fees: float = 0.0
    fixed_fees: float = 0.0
    slippage: float = 0.0
    min_size: float = 0.0
    max_size: float = np.inf
    size_granularity: float = np.nan
    reject_prob: float = 0.0
    lock_cash: bool = False
    allow_partial: bool = True
    raise_reject: bool = False
    log: bool = False
```

## NoOrder
```python
NoOrder = Order(
    size=np.nan,
    price=np.nan,
    size_type=-1,
    direction=-1,
    fees=np.nan,
    fixed_fees=np.nan,
    slippage=np.nan,
    min_size=np.nan,
    max_size=np.nan,
    size_granularity=np.nan,
    reject_prob=np.nan,
    lock_cash=False,
    allow_partial=False,
    raise_reject=False,
    log=False
)
```

## OrderResult
```python
class OrderResult(tp.NamedTuple):
    size: float
    price: float
    fees: float
    side: int
    status: int
    status_info: int
```

# 调整上下文

## AdjustSLContext
```python
class AdjustSLContext(tp.NamedTuple):
    i: int
    col: int
    position_now: float
    val_price_now: float
    init_i: int
    init_price: float
    curr_i: int
    curr_price: float
    curr_stop: float
    curr_trail: bool
```

## AdjustTPContext
```python
class AdjustTPContext(tp.NamedTuple):
    i: int
    col: int
    position_now: float
    val_price_now: float
    init_i: int
    init_price: float
    curr_stop: float
```

## SignalContext
```python
class SignalContext(tp.NamedTuple):
    i: int
    col: int
    position_now: float
    val_price_now: float
    flex_2d: bool
```

# 记录数据类型

## order_dt
```python
order_dt = np.dtype([
    ('id', np.int64),
    ('col', np.int64),
    ('idx', np.int64),
    ('size', np.float64),
    ('price', np.float64),
    ('fees', np.float64),
    ('side', np.int64),
], align=True)
```

## trade_dt
```python
_trade_fields = [
    ('id', np.int64),
    ('col', np.int64),
    ('size', np.float64),
    ('entry_idx', np.int64),
    ('entry_price', np.float64),
    ('entry_fees', np.float64),
    ('exit_idx', np.int64),
    ('exit_price', np.float64),
    ('exit_fees', np.float64),
    ('pnl', np.float64),
    ('return', np.float64),
    ('direction', np.int64),
    ('status', np.int64),
    ('parent_id', np.int64)
]
```

## log_dt
```python
_log_fields = [
    ('id', np.int64),
    ('group', np.int64),
    ('col', np.int64),
    ('idx', np.int64),
    ('cash', np.float64),
    ('position', np.float64),
    ('debt', np.float64),
    ('free_cash', np.float64),
    ('val_price', np.float64),
    ('value', np.float64),
    ('req_size', np.float64),
    ('req_price', np.float64),
    ('req_size_type', np.int64),
    ('req_direction', np.int64),
    ('req_fees', np.float64),
    ('req_fixed_fees', np.float64),
    ('req_slippage', np.float64),
    ('req_min_size', np.float64),
    ('req_max_size', np.float64),
    ('req_size_granularity', np.float64),
    ('req_reject_prob', np.float64),
    ('req_lock_cash', np.bool_),
    ('req_allow_partial', np.bool_),
    ('req_raise_reject', np.bool_),
    ('req_log', np.bool_),
    ('new_cash', np.float64),
    ('new_position', np.float64),
    ('new_debt', np.float64),
    ('new_free_cash', np.float64),
    ('new_val_price', np.float64),
    ('new_value', np.float64),
    ('res_size', np.float64),
    ('res_price', np.float64),
    ('res_fees', np.float64),
    ('res_side', np.int64),
    ('res_status', np.int64),
    ('res_status_info', np.int64),
    ('order_id', np.int64)
]
```